In [0]:
from config.f1_config import add_ingestion_metadata,CATALOG, BRONZE_SCHEMA, SILVER_SCHEMA
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType,TimestampType, FloatType
import pyspark.sql.functions as F

SILVER_TABLE = "f1_workspace.silver.races"

In [0]:
df_bronze_races = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.races")
display(df_bronze_races)

TRASFORM AND FLATTEN DATA


In [0]:
def transform_races_silver(df_bronze):
    
    return (
        df_bronze.select(
            F.trim(F.col("Circuit.circuitId")).alias("Circuit_Id"),
            F.trim(F.initcap(F.col("raceName"))).alias("Race_Name"),
            F.trim(F.col("season")).cast("int").alias("Season"),
            F.trim(F.col("round")).cast("int").alias("Round"),
            F.trim(F.col("time")).alias("Race_Time"),
            F.trim(F.col("date")).cast("timestamp").alias("Race_Date"),
        ) 
        .withColumn("silver_processed_timestamp", F.current_timestamp())
    )


df_silver_races = transform_races_silver(df_bronze_races)

display(df_silver_races)


In [0]:
df_silver_races.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", True)\
    .saveAsTable(SILVER_TABLE)